In [19]:
from huggingface_hub import snapshot_download

# Download model to a specific location\n",
model_path = snapshot_download(
    repo_id="nlptown/bert-base-multilingual-uncased-sentiment",
    cache_dir="./tmp/models"
)

# Then load from local path
from transformers import pipeline
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
    # local_files_only=True
)

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 27648.68it/s]
The tokenizer you are loading from './tmp/models/models--nlptown--bert-base-multilingual-uncased-sentiment/snapshots/8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


In [20]:
# Usage
result = classifier("This product is amazing!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

[{'label': '5 stars', 'score': 0.8754845857620239}]


In [21]:
model_path = snapshot_download(
    repo_id="cardiffnlp/twitter-roberta-base-sentiment",
    cache_dir="../../final_pipeline/models/overall_sentiment"
)

# Option 2: 3-class (positive/negative/neutral)
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
)

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 11875.15it/s]
Device set to use cuda:0


In [22]:
# Usage
result = classifier("This product is amazing!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

result = classifier("yesterday I bought an apple")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

result = classifier("This product is terrible!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

[{'label': 'LABEL_2', 'score': 0.9905126690864563}]
[{'label': 'LABEL_1', 'score': 0.5134272575378418}]
[{'label': 'LABEL_0', 'score': 0.9816111326217651}]


In [23]:
from transformers import pipeline
import torch.nn.functional as F
import torch

# Load with top_k=None to get probabilities for all 3 classes
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
    top_k=None 
)

def get_continuous_score(text):
    results = classifier(text)[0]
    
    # Map results to a dictionary for easy access
    # CardiffNLP labels: LABEL_0 (Neg), LABEL_1 (Neu), LABEL_2 (Pos)
    scores = {res['label']: res['score'] for res in results}
    
    # Calculate the weighted average
    # (Prob_Pos * 1) + (Prob_Neu * 0) + (Prob_Neg * -1)
    continuous_score = (scores['LABEL_2'] * 1) + (scores['LABEL_1'] * 0) + (scores['LABEL_0'] * -1)
    
    return continuous_score

# Examples:
print(f"Negative: {get_continuous_score('This is absolute garbage.')}") 
# Output approx: -0.98
print(f"Neutral:  {get_continuous_score('The package arrived on Tuesday.')}") 
# Output approx: 0.02
print(f"Positive: {get_continuous_score('I am so incredibly happy with this!')}") 
# Output approx: 0.95

Device set to use cuda:0


Negative: -0.9691423084586859
Neutral:  0.16997051052749157
Positive: 0.9911564703797922


In [24]:
# # Option 1: Binary (positive/negative)
# classifier = pipeline(
#     "sentiment-analysis",
#     model="distilbert-base-uncased-finetuned-sst-2-english"
# )

# # Option 2: 3-class (positive/negative/neutral)
# classifier = pipeline(
#     "sentiment-analysis",
#     model="cardiffnlp/twitter-roberta-base-sentiment"
# )

In [25]:
import pandas as pd

df_1 = pd.read_excel("../../data/initial_data/Study 1 reviews.xlsx")

df_1.columns

Index(['ID', 'finalReview', 'Satisfaction_final', 'cleaning_service_quality',
       'order_packaging', 'communication_and_responsiveness',
       'Driver_professionalism', 'Service_speed',
       'cleaning_service_quality_sentiment', 'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM'],
      dtype='object')

In [26]:
df_1["finalReview"] = df_1["finalReview"].fillna("").astype(str)

texts = df_1["finalReview"].tolist()

final_label = {'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive'}

results = classifier(texts, batch_size=32)
df_1["Overall_review_sentiment"] = [final_label[r["label"]] for r in results]


TypeError: list indices must be integers or slices, not str

In [ ]:
df_1.head()

,ID,finalReview,Satisfaction_final,cleaning_service_quality,order_packaging,communication_and_responsiveness,Driver_professionalism,Service_speed,cleaning_service_quality_sentiment,order_packaging_sentiment,communication_and_responsiveness_sentiment,Driver_professionalism_sentiment,Service_speed_sentiment,Overall_review_sentiment,Emotional_intensity_LLM
0,1.0,My order was to dryclean! All suits came back ...,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN
1,2.0,poor experience. jacket not cleaned properly.,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN
2,3.0,The clean laundry came in a bag that had a sme...,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN
3,4.0,not happy with the service. i received multipl...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN
4,5.0,its a very expensive service.,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN


### Continuous score from -1 to 1

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

model_path = snapshot_download(
    repo_id="cardiffnlp/twitter-roberta-base-sentiment",
    cache_dir="../../final_pipeline/models/overall_sentiment"
)

# Load the model but force it to have 1 output unit (Regression)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1, 
    ignore_mismatched_sizes=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Example usage
inputs = tokenizer("The product was okay, not great.", return_tensors="pt")
with torch.no_grad():
    output = model(**inputs)
    # This is your continuous score (unscaled)
    raw_score = output.logits.item() 

score = ((raw_score - 1) / (5 - 1)) * (1 - (-1)) + (-1)

print(f"Continuous Score: {score}")

OSError: PermissionError at /workspace when downloading nlptown/bert-base-multilingual-uncased-sentiment. Check cache directory permissions. Common causes: 1) another user is downloading the same model (please wait); 2) a previous download was canceled and the lock file needs manual removal.